# PHARVO-beta: POS Medicine Search Test

**Objective:** Verify that an authorized staff user (`rafi`) can access the **POS / Sales** terminal, search for a medicine (`Brufen`), and verify that matching catalog records appear with accurate **PC** pricing and stock availability.

### Test Criteria
- **Medicine Search Query:** `Brufen`
- **Verified Unit:** `PC`

### Prerequisites
```bash
pip install selenium webdriver-manager
```
Ensure PHARVO frontend is running at `http://localhost:5173` and backend at `http://localhost:8000`.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Configuration & Test Data ---
BASE_URL = "http://localhost:5173"
USERNAME = "rafi"
PASSWORD = "password"  # Replace with actual password
SEARCH_TERM = "Brufen"  # Medicine / Brand name (e.g., Brufen 400mg)
TARGET_UNIT = "PC"      # Unit to verify

# Step 1: Open browser and maximize window
driver = webdriver.Chrome()
driver.maximize_window()

# Set explicit wait helper (up to 10 seconds)
wait = WebDriverWait(driver, 10)

try:
    print(f"[INFO] Starting POS Medicine Search Test for '{SEARCH_TERM}'...")

    # Step 2: Open login page and sign in
    driver.get(f"{BASE_URL}/")

    username_field = wait.until(
        EC.visibility_of_element_located((By.ID, "username"))
    )
    password_field = wait.until(
        EC.visibility_of_element_located((By.ID, "password"))
    )

    username_field.clear()
    username_field.send_keys(USERNAME)

    password_field.clear()
    password_field.send_keys(PASSWORD)

    sign_in_button = wait.until(
        EC.element_to_be_clickable((By.ID, "sign-in-btn"))
    )
    sign_in_button.click()

    # Step 3: Navigate to POS / Sales module via sidebar
    pos_nav = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//aside//button[contains(., 'POS / Sales')]")
        )
    )
    pos_nav.click()

    # Verify POS screen title
    wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//header//h1[contains(text(), 'POS / Sales')]")
        )
    )
    print("[INFO] Navigated to POS / Sales terminal.")

    # Step 4: Locate POS medicine search bar
    # Selectors: aria-label="Search medicine or brand" or placeholder in CatalogSearch.jsx
    pos_search_input = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@aria-label='Search medicine or brand' and contains(@class, 'pos-input')]")
        )
    )

    # Step 5: Enter search query
    pos_search_input.clear()
    pos_search_input.send_keys(SEARCH_TERM)
    print(f"[INFO] Entered search query: '{SEARCH_TERM}'")

    # Step 6: Wait for filtered results in the POS Catalog table
    matching_cell = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, f"//table[contains(@class, 'pos-table')]//tr[contains(@class, 'pos-row-tr')]//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), '{SEARCH_TERM.lower()}')]")
        )
    )

    # Collect visible POS table rows
    result_rows = driver.find_elements(
        By.XPATH, "//table[contains(@class, 'pos-table')]//tbody//tr[contains(@class, 'pos-row-tr')]"
    )

    # Step 7: Verify result row and unit price details
    if result_rows:
        target_row = result_rows[0]
        med_name = target_row.find_element(By.XPATH, ".//td[1]//span[1]").text
        category = target_row.find_element(By.XPATH, ".//td[2]").text
        pc_price = target_row.find_element(By.XPATH, ".//td[3]").text
        stock_info = target_row.find_element(By.XPATH, ".//td[6]").text

        print(f"PASS: POS Medicine search successful for '{SEARCH_TERM}'.")
        print(f"      - Medicine: '{med_name}'")
        print(f"      - Category: '{category}'")
        print(f"      - {TARGET_UNIT} Price: '{pc_price}'")
        print(f"      - Current Stock: '{stock_info}'")

        # Step 8: Assert that the PC price is populated and not empty/missing
        assert pc_price and pc_price != "?", f"Expected valid {TARGET_UNIT} price, got '{pc_price}'"
        print(f"PASS: {TARGET_UNIT} unit price verified: {pc_price}")

    else:
        print(f"FAIL: No POS catalog records found matching '{SEARCH_TERM}'.")

except Exception as error:
    print(f"FAIL: POS Medicine Search test encountered error: {error}")

finally:
    # Step 9: Close browser
    print("[INFO] Cleaning up and closing browser...")
    driver.quit()
